# Gold Table: gold.video_type_summary

## Objective

Create an aggregated table to compare the performance of Shorts and Normal videos.

## Source
- gold.video_performance

## Grain
- One row per video type

## Group By

- video_type

## Columns

| Column | Description |
|---------|-------------|
| video_type | Short or Normal |
| total_videos | Number of videos |
| total_views | Sum of views |
| total_likes | Sum of likes |
| total_comments | Sum of comments |
| avg_views | Average views |
| avg_likes | Average likes |
| avg_comments | Average comments |
| avg_duration | Average video duration |

## Business Purpose

- Compare Shorts and Normal videos.
- Measure audience engagement by video type.
- Support summary dashboards and visual comparisons.

In [0]:
from pyspark.sql import functions as f 

In [0]:
silver_table="youtube.silver.yt_data"
gold_table="youtube.gold.video_type_summary"

In [0]:
video_type_summary_df = (
    spark.read.table(silver_table)
    .groupBy("video_type")
    .agg(
        f.sum("views").alias("total_views"),
        f.sum("likes").alias("total_likes"),
        f.sum("comments").alias("total_comments"),
        f.count("video_id").alias("total_videos"),
        f.round(f.avg("views"), 2).alias("avg_views"),
        f.round(f.avg("likes"), 2).alias("avg_likes"),
        f.round(f.avg("comments"), 2).alias("avg_comments"),
        f.round(f.avg(
            f.split("duration", ":")[0].cast("int") * 3600 +
            f.split("duration", ":")[1].cast("int") * 60 +
            f.split("duration", ":")[2].cast("int")
        ),2).alias("avg_duration_seconds"),
    )
)


In [0]:
(
    video_type_summary_df.write.mode("overwrite").saveAsTable(gold_table)
)

In [0]:
%sql
select * from youtube.gold.video_type_summary;